# Gate 2 (redo): evaluate θ_ref = ref_long on Colab

Run this after `stage1_long.ipynb` has finished (same or new session).
**Upload first:** `colab/code_bundle.tar.gz` to `MyDrive/unlearn/` (the data bundle is already there).
Evaluation is the same as for the pilot: 500 fixed chunks per split, 200 generations per language from
*validation* prompts. `pt_test` is used exactly once (logged in `results/test_access.jsonl`; a second
access is refused).

In [ ]:
# 1. Settings + GPU check
SEED = 0
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Mount Drive. MyDrive/unlearn/{checkpoints,results} is the working copy from Phase 3 on.
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/unlearn'
import os
for d in ('checkpoints', 'results', 'gate_packs'):
    os.makedirs(f'{DRIVE}/{d}', exist_ok=True)
print(sorted(os.listdir(DRIVE)))

In [ ]:
# 3. Data bundle (unchanged since Phase 0) + latest code bundle on top; check the data SHAs;
#    link checkpoints/ and results/ to Drive so every script writes there directly.
!rm -rf /content/unlearn && mkdir -p /content/unlearn
!tar xzf {DRIVE}/unlearn_bundle.tar.gz -C /content/unlearn
!tar xzf {DRIVE}/code_bundle.tar.gz -C /content/unlearn
%cd /content/unlearn
!cd data/processed && sha256sum -c --quiet MANIFEST.sha256 && echo "data OK"
!rm -rf checkpoints results && ln -s {DRIVE}/checkpoints checkpoints && ln -s {DRIVE}/results results

In [ ]:
# 4. Dependencies + tests. tests/test_data.py is not run here: it needs the *_titles.txt files, which are
#    not in the bundle. It passed on the Mac at Gate 0, and cell 3 already proved the .bin files are
#    byte-identical to the Mac copy (SHA-256). Smoke-data tests are skipped (no smoke data on Colab).
!pip -q install pyyaml
!python -m pytest -q --ignore=tests/test_data.py 2>&1 | tail -2

In [ ]:
# 5. Bring the Mac's results into Drive (never overwrites), then re-hash every protected checkpoint.
#    Drive cannot chmod, so this SHA check is what protects θ_ref. It must print OK, never raise.
!python scripts/colab_sync.py merge --src results_mac --dst results
!python scripts/colab_sync.py verify

In [ ]:
# 6. Training curves (train vs val bpc every 500 steps)
!python scripts/plots.py curves --runs stage1_long_s0
from IPython.display import Image; Image('results/figures/stage1_long_s0_curves.png')

In [ ]:
# 7. Evaluate θ_ref (skipped if already done: pt_test may be read only once)
import os
if os.path.exists(f'results/samples/ref_long_s{SEED}.json'):
    print('already evaluated')
else:
    !python scripts/eval_checkpoint.py --ckpt checkpoints/ref_long_s{SEED}.pt --name ref_long_s{SEED} \
        --splits pt_val en_val pt_train en_forget_train --test-splits pt_test --prompts val
!tail -1 results/eval.jsonl

In [ ]:
# Gate pack: re-check the hashes, then zip results/ (small files only, no checkpoints).
# Download MyDrive/unlearn/gate_packs/gate2_results.zip and unzip it into the project as colab_results/gate2/
!python scripts/colab_sync.py verify
!cd {DRIVE} && rm -f gate_packs/gate2_results.zip && zip -qr gate_packs/gate2_results.zip results && ls -la gate_packs